# 機器學習原理與技術

## 學習目標

完成本 Colab 後，你將能夠：

1. 說明機器學習中「特徵空間、標籤、模型、損失函數、評估指標」的關係。
2. 分辨監督式學習與非監督式學習的資料條件與任務目標。
3. 使用 `sklearn` 建立簡單的分類、迴歸、聚類與降維模型。
4. 觀察訓練集、測試集與評估指標如何影響模型解讀。
5. 透過小型實作理解 iPAS 中級考試常見的機器學習觀念。


In [ ]:
# ── 環境設定與範例資料 ───────────────────────────────
# 載入本章節會使用到的套件，並建立一組可重複使用的簡單分類資料。此資料可視為特徵空間中的點，每個樣本都有兩個特徵與一個類別標籤。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification

# 固定隨機種子，讓每次執行結果一致
np.random.seed(42)

X, y = make_classification(
    n_samples=120,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    class_sep=1.5,
    random_state=42
)

df = pd.DataFrame(X, columns=["feature_1", "feature_2"])
df["label"] = y

print("資料前 5 筆：")
print(df.head())
print("\n資料形狀：", df.shape)

plt.figure(figsize=(6, 4))
plt.scatter(df["feature_1"], df["feature_2"], c=df["label"], cmap="coolwarm", edgecolor="k")
plt.title("特徵空間中的分類資料")
plt.xlabel("feature_1")
plt.ylabel("feature_2")
plt.show()


## 核心概念說明

機器學習的基本流程可整理為：

1. **輸入資料與特徵空間**：原始資料經過前處理後，轉換成模型可理解的數值特徵。每筆資料可視為特徵空間中的一個點。
2. **任務目標與標籤型態**：若標籤是類別，通常是分類任務；若標籤是連續數值，通常是迴歸任務；若沒有標籤，常見任務包含聚類與降維。
3. **模型與假設空間**：模型是一個從輸入特徵映射到輸出結果的函數。不同模型代表不同假設，例如線性模型假設資料可用線性邊界描述。
4. **損失函數**：損失函數用來量化預測結果與真實答案之間的差距，例如分類常用交叉熵，迴歸常用 MSE 或 MAE。
5. **資料分割與評估**：訓練集用於學習參數，測試集用於檢查泛化能力。分類可用 Accuracy、Precision、Recall、F1；迴歸可用 MAE、MSE、R²。


In [ ]:
# ── 示範：監督式學習的分類任務 ───────────────────────────
# 這段程式碼示範如何將有標籤資料分成訓練集與測試集，並使用邏輯斯迴歸完成二元分類。重點是理解特徵、標籤、模型訓練與測試評估之間的關係。

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

X, y = make_classification(
    n_samples=160,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    class_sep=1.3,
    random_state=7
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7, stratify=y
)

model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred), 3))
print("Recall:", round(recall_score(y_test, y_pred), 3))
print("F1:", round(f1_score(y_test, y_pred), 3))
print("混淆矩陣：")
print(confusion_matrix(y_test, y_pred))

plt.figure(figsize=(6, 4))
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_pred, cmap="coolwarm", edgecolor="k")
plt.title("測試資料的預測分類結果")
plt.xlabel("feature_1")
plt.ylabel("feature_2")
plt.show()


## 任務類型比較

**監督式學習**使用具標籤資料訓練模型，目標是學會輸入與輸出之間的對應關係。常見任務包含：

- 分類：預測離散類別，例如垃圾信件偵測、疾病類型判斷。
- 迴歸：預測連續數值，例如房價、銷售量、滿意度分數。

**非監督式學習**不依賴明確標籤，目標是發現資料內部結構。常見任務包含：

- 聚類：依相似度自動分群，例如顧客分群。
- 降維：保留主要結構並降低特徵數，例如將高維資料壓縮成 2D 方便視覺化。

考試常見判斷方式：先看資料是否有標籤，再看輸出目標是類別、數值、群組還是低維表示。


In [ ]:
# ── 示範：非監督式學習的聚類與降維 ─────────────────────────
# 這段程式碼示範 KMeans 聚類與 PCA 降維。資料本身沒有提供標籤時，模型會根據樣本間的距離與分布結構自動找出群組。

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

X, _ = make_blobs(
    n_samples=180,
    centers=3,
    n_features=4,
    cluster_std=1.0,
    random_state=10
)

kmeans = KMeans(n_clusters=3, random_state=10, n_init=10)
cluster_labels = kmeans.fit_predict(X)
score = silhouette_score(X, cluster_labels)

pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)

print("聚類標籤前 10 筆：", cluster_labels[:10])
print("Silhouette score:", round(score, 3))
print("PCA 解釋變異比例：", np.round(pca.explained_variance_ratio_, 3))

plt.figure(figsize=(6, 4))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=cluster_labels, cmap="viridis", edgecolor="k")
plt.title("KMeans 聚類後的 PCA 2D 視覺化")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()


## 損失函數與評估指標

損失函數是模型學習的方向盤，用來告訴模型目前預測有多差。評估指標則用來描述模型在資料上的實際表現。

常見配對如下：

- 分類任務：可使用交叉熵作為訓練損失，並用 Accuracy、Precision、Recall、F1 評估。
- 迴歸任務：可使用 MSE 或 MAE 作為損失或評估指標。
- 聚類任務：因為沒有標準答案，常用 Silhouette score 等內部指標觀察群內相似與群間分離程度。

實務上不應只看單一指標。例如在詐欺偵測或醫療診斷中，Accuracy 可能很高，但 Recall 太低會漏掉重要案例。


In [ ]:
# ── 🧪 自我測驗 ──────────────────────────────────
# 請依照 TODO 註解完成填空，練習判斷任務類型、建立模型，並輸出對應評估結果。此版本已放入參考答案，建議先閱讀題目後再嘗試修改數值或模型。

import numpy as np
from sklearn.datasets import make_regression, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score

# 題目 1：迴歸任務
# TODO: 建立一組連續數值標籤資料，並使用線性迴歸模型預測 y
# Expected: 使用 make_regression 產生 X_reg, y_reg，y_reg 為連續數值標籤
X_reg, y_reg = make_regression(
    n_samples=120,
    n_features=3,
    noise=12,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=42
)

# TODO: 選擇適合「連續數值預測」的模型
# Expected: 使用 LinearRegression，並輸出 MAE、MSE、R2 作為迴歸評估指標
reg_model = LinearRegression()
reg_model.fit(X_train, y_train)
y_pred = reg_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("迴歸任務評估")
print("MAE:", round(mae, 2))
print("MSE:", round(mse, 2))
print("R2:", round(r2, 3))

# 題目 2：非監督式聚類任務
# TODO: 建立一組沒有標籤的群集資料，並使用 KMeans 找出群組
# Expected: 使用 make_blobs 產生 X_cluster，使用 KMeans(n_clusters=3) 取得 cluster_labels
X_cluster, _ = make_blobs(
    n_samples=150,
    centers=3,
    n_features=2,
    cluster_std=0.9,
    random_state=42
)

cluster_model = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = cluster_model.fit_predict(X_cluster)

# TODO: 選擇適合「無標籤聚類」的評估指標
# Expected: 使用 silhouette_score，分數越高通常代表群內越集中、群間越分離
sil_score = silhouette_score(X_cluster, cluster_labels)

print("\n聚類任務評估")
print("群組標籤前 10 筆：", cluster_labels[:10])
print("Silhouette score:", round(sil_score, 3))
